In [22]:
import polars as pl
import numpy as np
import datetime as dt
import os
import sys
import json 
import importlib

sys.path.append("../utils/")

import helpers as hp

In [2]:
rng = np.random.default_rng(seed=274)

In [18]:

with open("../configs/zone_distances.json", "r") as json_file:
    zone_distances = json.load(json_file)

In [20]:
zone_distances

{'Yerbabuena': {'centroids': [20.964404421308334, -101.2847459818324],
  'points': [{'point': [20.9743, -101.300626],
    'distance': 0.01871089133756122},
   {'point': [20.946278, -101.298483], 'distance': 0.022743632462390598},
   {'point': [20.951706, -101.264939], 'distance': 0.023527992541503697},
   {'point': [20.979447, -101.274624], 'distance': 0.018131014585796808},
   {'point': [20.985938, -101.285636], 'distance': 0.02155196379935847},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122},
   {'point': [20.9743, -101.300626], 'distance': 0.01871089133756122}]},
 'Marfil': {'centroids': [20.998877268148007, -101.29015919031652],
  'points': [{'point': [20.979212, -101.292779],
    'distance': 0.019839006379117525},
   {'point': [20.991242, -101.304647], 'distance': 0.016376628136369357},
   {'point': [21.006043, -101.294156], 'distance': 0.008205010702043177},
   {'point': [21.019272, -101.

In [10]:
gym_hours = {"open" : "6.0",
             "close" : "22.0"}

profiles = {
"1": {
"name" : "frequent_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .89},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .76},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .85},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .7},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.9},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .9},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "6.43",
"hour_mean" : ".75"
},
"2" : {
"name" : "frequent_noon",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .7},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .82},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .67},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .87},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.78},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .84},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "13.16",
"hour_mean" : "1.40"
            },
"3" :{
"name" : "frequent_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .87},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .76},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .84},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "19.5",
"hour_mean" : "1.15"
},
"4" :{
"name" : "random_morning",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .74},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .83},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .68},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 0.86},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "8.5",
"hour_mean" : "2.05"
},
"5" :{
"name" : "random_night",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .8},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : 1},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .8},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.4},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .8},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "20.1",
"hour_mean" : "2.08"
},
"6" :{
"name" : "full_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .5},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .5},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .5},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : 5},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.5},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .5},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.00",
"hour_mean" : "3.0"
},
"7" :{
"name" : "rare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .2},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .4},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .2},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.3},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .4},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "16.0",
"hour_mean" : "3.5"
},
"8" :{
"name" : "ultrarare_random",
"days_probability" : {"mon": {"day_number" : 1, "day_name" : "Monday", "day_weight" : .1},
                      "tue": {"day_number" : 2, "day_name" : "Tuesday", "day_weight" : .2},
                      "wed": {"day_number" : 3, "day_name" : "Wednesday", "day_weight" : .1},
                      "thu": {"day_number" : 4, "day_name" : "Thursday", "day_weight" : .1},
                      "fri": {"day_number" : 5, "day_name" : "Friday", "day_weight" :.2},
                      "sat": {"day_number" : 6, "day_name" : "Saturday", "day_weight" : .3},
                      "sun": {"day_number" : 7, "day_name" : "Sunday", "day_weight" : 0}
},
"hour_mean" : "14.5",
"hour_mean" : "5.5"
}
}

profile_weights = {"1" : 0.23076923, "2": 0.12820513, "3" : 0.25641026, "4" : 0.07692308, "5" : 0.07692308, "6" : 0.05128205, "7" : 0.07692308, "8" : 0.1025641}

for profile in profiles.keys():
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
        
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    
    profiles[profile]["total_weight"] = sum_weight

profiles

{'1': {'name': 'frequent_morning',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
    'day_weight': 0.89,
    'day_probability': 0.178},
   'tue': {'day_number': 2,
    'day_name': 'Tuesday',
    'day_weight': 0.76,
    'day_probability': 0.152},
   'wed': {'day_number': 3,
    'day_name': 'Wednesday',
    'day_weight': 0.85,
    'day_probability': 0.16999999999999998},
   'thu': {'day_number': 4,
    'day_name': 'Thursday',
    'day_weight': 0.7,
    'day_probability': 0.13999999999999999},
   'fri': {'day_number': 5,
    'day_name': 'Friday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sat': {'day_number': 6,
    'day_name': 'Saturday',
    'day_weight': 0.9,
    'day_probability': 0.18},
   'sun': {'day_number': 7,
    'day_name': 'Sunday',
    'day_weight': 0,
    'day_probability': 0.0}},
  'hour_mean': '.75',
  'total_weight': 5.0},
 '2': {'name': 'frequent_noon',
  'days_probability': {'mon': {'day_number': 1,
    'day_name': 'Monday',
   

In [131]:
# Functions
def create_customer_profiles(profile_weights_ : dict, n_):
    return {str(i): str(int(rng.choice( list(profile_weights.keys()), p = list(profile_weights.values()) ) ) ) for i in range(n_) }

In [132]:
create_customer_profiles(profiles, 9)

{'0': '8',
 '1': '7',
 '2': '1',
 '3': '5',
 '4': '8',
 '5': '5',
 '6': '5',
 '7': '1',
 '8': '1'}

In [12]:
# prob = np.array((.9,0.5,1,.3,.3,.2,.3,.4))

# prob/sum(prob)

In [135]:
importlib.reload(hp)

oli =hp.create_customer(zone_distances)

Creating customer
36254


In [136]:
oli

{'id': None,
 'name': 'Marisela Violeta',
 'last_name': 'Valdivia Ruiz',
 'gender': np.False_,
 'birth_date': datetime.datetime(2003, 9, 7, 0, 0),
 'lat': 20.987860256811654,
 'lon': -101.29836796811152,
 'zipcode': 36254,
 'email': 'r********w@hotmail.com',
 'phone_number': '(473)3387234',
 'created_at': datetime.datetime(2026, 5, 20, 20, 7, 23, 304311),
 'status': 'Active',
 'profile_type': None,
 'updated_at': datetime.datetime(2026, 5, 20, 20, 7, 23, 304317)}

In [ ]:
date = '2026-01-01'

def create_customer_access_data(date_, profile_metadata_, gym_hours_, customers_path_ = "../data/", customers_file_ = "customers.csv"):
    
    customers_schema = {"Id_Customer" :pl.Int64,
                        "Name": pl.String,
                        "Last_Name": pl.String,
                        "Gender" : pl.Boolean,
                        "Birth_Date" : pl.Datetime,
                        "Latitude": pl.Float32,
                        "Longitude" : pl.Float32,
                        "Zipcode" : pl.Int64 ,
                        "Email" : pl.String,
                        "Phone_Number" : pl.String,
                        "Created_At" : pl.Datetime,
                        "Status" : pl.String,
                        "Profile_Type" : pl.String,
                        "Updated_At" : pl.Datetime}

    eval_date = dt.datetime.strptime(date_, "%Y-%m-%d")
    day_names_dic = {0 : 'Monday', 1: 'Tuesday', 2: 'Wednesday', 3: 'Thursday', 4:'Friday', 5:'Saturday', 6: 'Sunday'}
    day_of_week = eval_date.weekday()
    day_name = day_names_dic[day_of_week]

    print(eval_date, day_of_week, day_name)
    
    if not os.path.isdir(customers_path_):
        os.makedirs(customers_path_, exist_ok = True)

    if not os.path.exists(customers_path_ + customers_file_):
        print("Creating customers file")
        df_customers = pl.DataFrame([], schema = customers_schema,
                                                    orient="row").write_csv(customers_path_+ customers_file_)
    
    df_customers = pl.read_csv(customers_path_+ customers_file_, schema=customers_schema)

    # New customers
    tot_new = int(abs(rng.normal(0, 1)))
    print("Total new customers", tot_new)

    max_id = df_customers.select(pl.max("Id_Customer")).item()

    counter = max_id
    if max_id == None:
        counter = 1

    dict_profiles = create_customer_profiles(profile_metadata_, tot_new)

    new_customers = []
    for new_cus in range(tot_new):

        new_customer = hp.create_customer(zone_distances)
        new_customer["id"] = counter
        new_customer["profile_type"] = dict_profiles[str(new_cus)]
        new_customer["created_at"] = dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

        new_customers.append(tuple(new_customer.values()))
        counter +=1

        
    df_new_customers = pl.DataFrame(new_customers, customers_schema, orient = "row")

    df_customers = pl.concat([df_customers, df_new_customers])

    df_customers.write_csv(customers_path_+customers_file_)

    if df_customers.shape[0] >0:
        temp = df_customers.filter(pl.col("Status") =="Active").to_dicts()

        for row in temp:
            profile = str(row["Profile_Type"])
            profile_data = profile_metadata_[profile]
            day_proba  = profile_data["days_probability"][day_name[0:3].lower()]["day_probability"]

            has_visit = rng.choice([1,0], p = [day_proba, 1-day_proba])
            print(profile, day_proba, has_visit)


    return df_customers
    

create_customer_access_data(date, profiles, gym_hours)

2026-01-01 00:00:00 3 Thursday
Total new customers 0
8 0.1 0


Id_Customer,Name,Last_Name,Gender,Birth_Date,Latitude,Longitude,Zipcode,Email,Phone_Number,Created_At,Status,Profile_Type,Updated_At
i64,str,str,bool,datetime[μs],f32,f32,i64,str,str,datetime[μs],str,str,datetime[μs]
1,"""Benito Gabino""","""Maldonado Otero""",true,2004-08-20 00:00:00,21.017767,-101.254303,36000,"""a**********e@hotmail.com""","""(473)7984438""",2026-01-01 20:47:13.525995,"""Active""","""8""",2026-05-20 20:47:13.525957


In [ ]:
for profile in profiles.keys():
    days = profiles[profile]["days_probability"]

    sum_weight = sum([days[day]["day_weight"] for day in days.keys()])
    
    for day in days.keys():
        days[day]["day_probability"] = days[day]["day_weight"] /sum_weight
    profiles[profile]["total_weight"] = sum_weight
profiles
        

In [143]:
dt.datetime.combine(dt.datetime.strptime(date, "%Y-%m-%d"), dt.datetime.now().time())

datetime.datetime(2026, 1, 1, 20, 39, 30, 722140)

In [121]:
my_dic = {}


for i in range(100):
    value = int(abs(rng.normal(0, 1.5)))
    my_dic[value] = my_dic.get(value, 0) + 1

my_dic

{0: 49, 1: 33, 2: 12, 4: 2, 3: 3, 5: 1}